In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
from codes import OpticalData, Fault
import numpy as np

optical = OpticalData(ew_filepath="/Users/hintont/Dev/projects/Ridgecrest/data/EW_Ridgecrest_1m_utm.tif", ns_filepath="/Users/hintont/Dev/projects/Ridgecrest/data/NS_Ridgecrest_1m_utm.tif", verbose=False)
optical = optical.decimate(100)
optical = optical.clear_nan()

fault = Fault(filepath="/Users/hintont/Dev/projects/Ridgecrest/data/fault_trace.shp", crs=optical.ew.rio.crs)
profiles = fault.gen_profiles()
profiles = optical.evaluate_profiles(profiles)

profile = profiles[3]

xs = profile.xs
data = profile.displacements[1]
noise_level=0.05
data_covariance = noise_level**2 * np.eye(xs.size)

In [ ]:
from codes import HamiltonianInversion, UniformDist, TwoDDzForwardModel

n_patches = 12

priors = [UniformDist("dz_halfwidth", 0., 1000.)] + [UniformDist(f"slip{i}", -10, 0) for i in range(n_patches)]

forward_model = TwoDDzForwardModel()
forward_model = forward_model.build_uniform_patches(n_patches, 12000.)
forward_model.xs = xs

inversion = HamiltonianInversion(forward_model, priors, data, data_covariance)
inversion = inversion.run()

Starting inversion...
[<codes.Dist.UniformDist object at 0x11f5e3d90>, <codes.Dist.UniformDist object at 0x11f5e3b10>, <codes.Dist.UniformDist object at 0x11f5bba80>, <codes.Dist.UniformDist object at 0x11f5bb950>, <codes.Dist.UniformDist object at 0x1112a43b0>, <codes.Dist.UniformDist object at 0x11f5f79b0>, <codes.Dist.UniformDist object at 0x11f5f7790>, <codes.Dist.UniformDist object at 0x11f68d450>, <codes.Dist.UniformDist object at 0x11f68dc50>, <codes.Dist.UniformDist object at 0x1105418b0>, <codes.Dist.UniformDist object at 0x11f5eaa80>, <codes.Dist.UniformDist object at 0x11fff7bd0>, <codes.Dist.UniformDist object at 0x11fff7a10>]
Linear estimation of gradient in Greens function (13)
p0 p1 p2 p3 p4 p5 p6 p7 p8 p9 p10 p11 p12 
... estimation complete:


/Users/hintont/Dev/codes/.venv/lib/python3.14/site-packages/pytensor/tensor/blockwise.py:514: RuntimeWarning: divide by zero encountered in matmul
  return (core_func(*inputs),)
/Users/hintont/Dev/codes/.venv/lib/python3.14/site-packages/pytensor/tensor/blockwise.py:514: RuntimeWarning: overflow encountered in matmul
  return (core_func(*inputs),)
/Users/hintont/Dev/codes/.venv/lib/python3.14/site-packages/pytensor/tensor/blockwise.py:514: RuntimeWarning: invalid value encountered in matmul
  return (core_func(*inputs),)
Initializing NUTS using jitter+adapt_diag...
/Users/hintont/Dev/codes/.venv/lib/python3.14/site-packages/pytensor/tensor/blockwise.py:514: RuntimeWarning: divide by zero encountered in matmul
  return (core_func(*inputs),)
/Users/hintont/Dev/codes/.venv/lib/python3.14/site-packages/pytensor/tensor/blockwise.py:514: RuntimeWarning: overflow encountered in matmul
  return (core_func(*inputs),)
/Users/hintont/Dev/codes/.venv/lib/python3.14/site-packages/pytensor/tensor/bl

/Users/hintont/Dev/codes/.venv/lib/python3.14/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 2 seconds.


In [10]:
inversion.result["mean"][1:].to_numpy()

array([-3.   , -2.999, -2.998, -2.998, -2.997, -2.995, -2.993, -2.991,
       -2.989, -2.987, -2.984, -2.982])